# 📋 [OPS08_M02] 관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m02_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M02
- **미션 주제:** 관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 02. 관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"M1 역할분담 매트릭스를 바탕으로 관측자·지휘자·경계조의 단계별 임무를 "신호-행동-보고" 3요소 표로 변환하고, 8대 필수 안전/보안 수칙을 적절한 단계에 배치해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 사격 전 지휘자 결심 및 교전수칙(ROE) 확인 단계가 명시되었는지, 모호한 표현(신속히/적절히)이 정량화되었는지 검토합니다.

---

### 📑 실습자료 및 과업 지침

관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성**

가. 제공 자료 - 필수 안전/보안 조치 8개 [교육용가상자료]

| **번호** | **필수 조치** | **배치할 단계(작성)** |
| --- | --- | --- |
| **S1** | 경보 수신 즉시 지정된 은폐 위치로 이동한다(긴급 모드). | ( ) |
| **S2** | 사격은 지휘자 결심 후에만 한다. 훈련에서는 구두 선언으로 대체한다. | ( ) |
| **S3** | 사격 전 사격 방향에 아군·민간인이 없는지 확인한다(오사격 방지). | ( ) |
| **S4** | 무전은 짧게 하고 평문으로 진지 위치를 말하지 않는다. | ( ) |
| **S5** | 대피 후 지휘자는 인원을 확인(점호)하고 결과를 보고한다. | ( ) |
| **S6** | 부상자가 생기면 즉시 보고하고 응급처치·의무 인원에 연계한다. | ( ) |
| **S7** | 경보 해제 명령 전에는 임의로 원위치로 복귀하지 않는다. | ( ) |
| **S8** | 경보 중 개인 휴대전화·전자기기 사용과 촬영을 금지한다. | ( ) |

나. 제공 자료 - 모호 표현 바꾸기 예시

| **모호한 표현** | **구체적 기준(예시)** |
| --- | --- |
| **신속히 보고한다** | 포착 후 30초 이내에 방위·수량만이라도 1차 보고한다 |
| **적절한 위치로 대피한다** | 사전 지정한 은폐 위치 ○번(가상)으로 이동한다 |
| **상황을 판단해 조치한다** | 2명 이상 확인되면 경보 발령, 1명만 확인되면 '확인 요망' 전파 |
| **필요시 사격한다** | 지휘자의 '사격' 명령이 있을 때만 구두 사격 선언 |

다. 작성 서식 1 - 역할별 표준 행동 지침(SOP) 표 (신호 - 행동 - 보고)

| **단계** | **신호(무엇을 보거나 들으면)** | **관측자** | **지휘자** | **경계조** | **안전 조치(S번호)** |
| --- | --- | --- | --- | --- | --- |
| **① 포착** |  |  |  |  |  |
| **② 식별** |  |  |  |  |  |
| **③ 보고** |  |  |  |  |  |
| **④ 전파** |  |  |  |  |  |
| **⑤ 결심** |  |  |  |  |  |
| **⑥ 대피·대응** |  |  |  |  |  |
| **⑦ 해제** |  |  |  |  |  |

라. 작성 서식 2 - SOP 흐름도 틀 (빈칸을 채워 완성하고, 긴급/일반 분기를 표시한다)

| [① 포착: 소리 또는 육안]
 \|
 < 즉시 위협인가? (근접·급강하·다수) >
 \|예 \|아니오
 [긴급 모드] [일반 모드]
 전원 ( ) 관측자 ( )
 지휘자 ( ) 지휘자 복창·교차 확인 ( )
 \| \|
 +------------+---------------+
 \|
 [⑤ 결심: ( )]
 \|
 [⑥ 대피·대응: ( )]
 \|
 [⑦ 해제: 해제 명령 → 인원 확인 → ( )] |
| --- |

마. 작성 서식 3 - 휴대용 요약 카드(Pocket Guide, 명함 2장 크기로 줄여 인쇄)

| **구분** | **관측자** | **지휘자** | **경계조** |
| --- | --- | --- | --- |
| **긴급 모드 첫 행동** | ( ) | ( ) | ( ) |
| **일반 모드 첫 행동** | ( ) | ( ) | ( ) |
| **반드시 말할 것** | ( ) | ( ) | ( ) |
| **절대 하지 말 것** | ( ) | ( ) | ( ) |

바. 교차 검토 기록표 (다른 조가 작성)

| **번호** | **지적 위치** | **지적 내용(누락·모호)** | **수정 결과** | **확인** |
| --- | --- | --- | --- | --- |
| **1** |  |  |  |  |
| **2** |  |  |  |  |
| **3** |  |  |  |  |
| **4** |  |  |  |  |
| **5** |  |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 7단계 모두 관측자·지휘자·경계조 행동이 적혔다(빈칸 0) | ( ) |
| 필수 안전/보안 조치 S1~S8이 모두 배치되었다 | ( ) |
| 사격 행동은 지휘자 결심 이후에만 나온다 | ( ) |
| 긴급/일반 모드가 흐름도에서 구분된다 | ( ) |
| 교차 검토 지적 사항을 모두 수정했다 | ( ) |
| 요약 카드가 한 장 안에 들어간다 | ( ) |

**

